In [0]:
import urllib.request

url = 'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz'
local_path = "/tmp/products_raw.csv.gz"
adls_raw_path = "abfss://artgulidov@dlsua5816bd.dfs.core.windows.net/staging/"
csv_raw_name = "products_raw.csv.gz"

try:
    urllib.request.urlretrieve(url, local_path)
    print(f"File successfully downloaded to {local_path}")
except Exception as e:
    print(f"An error occurred: {e}")

dbutils.fs.cp(
    f"file:{local_path}",
    adls_raw_path + csv_raw_name
)

In [0]:
df = spark.read \
    .option("header", True) \
    .option("delimiter", "\t") \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("compression", "gzip") \
    .csv(adls_raw_path + csv_raw_name)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("path", adls_raw_path)
    .saveAsTable("dbr_dev_ua5816bd.gulidov_artem_staging.off_staging")
)